# Tag 09 — Fortgeschrittene  
## StackingClassifier — Steel Industry Energy Dataset

نسخه اصلاح‌شده با:
- نام دقیق فایل `Steel_industry_data.csv`
- اصلاح نام ستون‌ها
- `SimpleImputer` برای رفع خطای `NaN`
- اجرای سبک‌تر با `sample_size=1000`
- ساخت خودکار `comparison_df`
- خروجی‌های آماده برای گزارش

## Dataset Validation
Run this cell to verify that your datasets are present and correctly formatted.

In [ ]:
# --- DATASET VALIDATION ---
import os
import pandas as pd

def validate_dataset(filepath, expected_columns=None, avoid_columns=None):
    if not os.path.exists(filepath):
        print(f'❌ ERROR: Dataset not found at {filepath}')
        return False
    try:
        df = pd.read_csv(filepath, nrows=5)
        print(f'✅ SUCCESS: Dataset found at {filepath} (Columns: {df.shape[1]})')
        if expected_columns:
            missing = [c for c in expected_columns if c not in df.columns]
            if missing:
                print(f'⚠️ WARNING: Missing expected columns: {missing}')
                return False
        if avoid_columns:
            forbidden = [c for c in avoid_columns if c in df.columns]
            if forbidden:
                print(f'❌ ERROR: Found forbidden columns {forbidden}. Wrong dataset!')
                return False
        return True
    except Exception as e:
        print(f'❌ ERROR: Could not read dataset: {e}')
        return False

print('Validating Ensemble Learning Datasets...')
validate_dataset('../data/winequality-red.csv')
validate_dataset('../data/Steel_industry_data.csv')

In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

print("Imports OK ✅")

Imports OK ✅


## 1. Load data

In [2]:
PROJECT_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path(r"C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\Tag_09_Ensemble_Methods_Project")
]

DATA_PATH = None
for base in PROJECT_DIR_CANDIDATES:
    candidate = base / "data" / "raw" / "Steel_industry_data.csv"
    if candidate.exists():
        DATA_PATH = candidate
        PROJECT_DIR = base
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Steel_industry_data.csv not found. Put it in: "
        "Tag_09_Ensemble_Methods_Project/../data/Steel_industry_data.csv"
    )

OUTPUT_DIR = PROJECT_DIR / "output" / "Fortgeschrittene"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Using data file:", DATA_PATH)

try:
    df = pd.read_csv(DATA_PATH)
except UnicodeDecodeError:
    df = pd.read_csv(DATA_PATH, encoding="latin1")

df.columns = df.columns.str.strip()

rename_map = {
    "date": "Datum",
    "Date": "Datum",
    "WeekStatus": "wochenstatus",
    "Week_Status": "wochenstatus",
    "Day_of_week": "wochentag",
    "DayOfWeek": "wochentag",
    "Load_Type": "lasttyp",
    "LoadType": "lasttyp"
}

df = df.rename(columns=rename_map)

required_cols = ["lasttyp"]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(
        f"Missing target column: {missing}\n"
        f"Available columns: {df.columns.tolist()}"
    )

print("Shape:", df.shape)
print(df.head())
print(df.columns.tolist())

FileNotFoundError: Steel_industry_data.csv not found. Put it in: Tag_09_Ensemble_Methods_Project/../data/Steel_industry_data.csv

## 2. Prepare X and y

In [3]:
df = df.dropna(subset=["lasttyp"]).copy()

cols_to_drop = [col for col in ["Datum", "lasttyp"] if col in df.columns]
X = df.drop(columns=cols_to_drop)
y = df["lasttyp"]

print("X shape:", X.shape)
print("y distribution:")
print(y.value_counts())
print("Missing values in X:", int(X.isna().sum().sum()))

NameError: name 'df' is not defined

## 3. Train/test split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

NameError: name 'X' is not defined

## 4. Preprocessor with Imputer

In [5]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", encoder)
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Preprocessor is ready ✅")

NameError: name 'X_train' is not defined

## 5. Cross-validation for base learners

In [6]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

sample_size = min(1000, len(X_train))
X_train_small = X_train.sample(n=sample_size, random_state=42)
y_train_small = y_train.loc[X_train_small.index]

base_learners_for_cv = {
    "DecisionTree": DecisionTreeClassifier(max_depth=8, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=80, random_state=42, n_jobs=1),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "KNeighbors": KNeighborsClassifier(n_neighbors=5)
}

rows = []
for name, estimator in base_learners_for_cv.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    scores = cross_val_score(
        pipe,
        X_train_small,
        y_train_small,
        cv=cv,
        scoring="accuracy",
        n_jobs=1
    )
    rows.append({
        "model": name,
        "cv_accuracy_mean": scores.mean(),
        "cv_accuracy_std": scores.std()
    })

base_cv_df = pd.DataFrame(rows).sort_values("cv_accuracy_mean", ascending=False).reset_index(drop=True)
base_cv_df.to_csv(OUTPUT_DIR / "intermediate_base_models_cv_accuracy.csv", index=False)
base_cv_df

NameError: name 'X_train' is not defined

## 6. StackingClassifier with Logistic Regression meta model

In [7]:
base_learners = [
    ("dt", DecisionTreeClassifier(max_depth=8, random_state=42)),
    ("rf", RandomForestClassifier(n_estimators=80, random_state=42, n_jobs=1)),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
    ("gb", GradientBoostingClassifier(random_state=42))
]

stack_logreg = StackingClassifier(
    estimators=base_learners,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=3,
    n_jobs=1
)

stack_logreg_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", stack_logreg)
])

stack_scores = cross_val_score(
    stack_logreg_pipe,
    X_train_small,
    y_train_small,
    cv=cv,
    scoring="accuracy",
    n_jobs=1
)

stack_logreg_result = pd.DataFrame([{
    "model": "Stacking_LogisticMeta",
    "cv_accuracy_mean": stack_scores.mean(),
    "cv_accuracy_std": stack_scores.std()
}])

stack_logreg_result.to_csv(OUTPUT_DIR / "intermediate_stacking_logistic_meta_cv_accuracy.csv", index=False)
stack_logreg_result

NameError: name 'preprocessor' is not defined

## 7. Compare base learners and stacking

In [8]:
comparison_df = pd.concat(
    [base_cv_df, stack_logreg_result],
    ignore_index=True
).sort_values("cv_accuracy_mean", ascending=False).reset_index(drop=True)

comparison_df.to_csv(OUTPUT_DIR / "intermediate_model_comparison_cv_accuracy.csv", index=False)
display(comparison_df)

plt.figure(figsize=(10, 5))
plt.bar(
    comparison_df["model"],
    comparison_df["cv_accuracy_mean"],
    yerr=comparison_df["cv_accuracy_std"],
    capsize=4
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("CV Accuracy")
plt.title("Base Learners vs StackingClassifier")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "intermediate_base_vs_stacking_cv_accuracy.png", dpi=200, bbox_inches="tight")
plt.show()

NameError: name 'base_cv_df' is not defined

## 8. Final test evaluation for StackingClassifier

In [9]:
stack_logreg_pipe.fit(X_train, y_train)
y_pred = stack_logreg_pipe.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
print(f"Stacking test accuracy: {test_acc:.4f}")

pd.DataFrame([{
    "model": "Stacking_LogisticMeta",
    "test_accuracy": test_acc
}]).to_csv(OUTPUT_DIR / "intermediate_stacking_test_accuracy.csv", index=False)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=stack_logreg_pipe.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=stack_logreg_pipe.classes_)

disp.plot(cmap="Blues", values_format="d")
plt.title("StackingClassifier - Test Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "intermediate_stacking_test_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

NameError: name 'stack_logreg_pipe' is not defined

## 9. Output files

In [10]:
print("Saved outputs in:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob("intermediate_*")):
    print(path.name)

NameError: name 'OUTPUT_DIR' is not defined